In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
current_dir = Path(os.getcwd())
if current_dir.name == "preprocessing":
    project_root = current_dir.parent
else:
    project_root = current_dir
    
input_path = project_root / 'data' / 'Speed Dating Data.csv'
output_path = project_root / 'data' / 'Columbia_filtered.csv'

df = pd.read_csv(input_path, encoding='latin-1')

In [3]:
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (8378, 195)


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


These are the columns that are also in the HCMST. 

In [ ]:
target_columns_dict = {
    'age': "Subject age",
    'age_o': "Partner age",
    'race': "Subject race",
    'race_o': "Partner race",  
    'gender': "Subject gender",
    'partner_gender': "Partner gender", 
    'attr': "Attraction",
    'Interracial_relationship': 'Interracial relationship', 
    'match' : 'match',
    'dec': "Subject dec", 
    'dec_o' : 'Partner dec', 
}

### Creating a new column "interracial_relationship" with 0 for not interracial, 1 for yes interracial
this format matches what is in the HCMST dataset

Checking if it is a interracial relationship. i.e: do their races match?



In [90]:
df['Interracial_relationship'] = (df['race'] != df['race_o']).astype('Int64')
df[['race','race_o', 'Interracial_relationship']]

/var/folders/wt/tgvfpnsd43jcb_s909_9lprr0000gn/T/ipykernel_3588/1451764418.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Interracial_relationship'] = (df['race'] != df['race_o']).astype('Int64')


,race,race_o,Interracial_relationship
0,4.0,2.0,1
1,4.0,2.0,1
2,4.0,4.0,0
3,4.0,2.0,1
4,4.0,3.0,1
...,...,...,...
8373,2.0,3.0,1
8374,2.0,6.0,1
8375,2.0,3.0,1
8376,2.0,4.0,1


In [ ]:
race_mapping = {
    1: 'BLACK',
    2: 'WHITE',
    3: 'HISPANIC',
    4: 'ASIAN',
    5: 'NATIVE AMERICAN',
    6: 'OTHER'
}

df["race"] = df["race"].map(race_mapping)
df["race_o"] = df["race_o"].map(race_mapping)

### Create a partner_gender column

this dataset does not have it.

In [92]:
df['partner_gender'] = np.where(df['gender'] == 0, 1, 0)

df[['gender','partner_gender']]

/var/folders/wt/tgvfpnsd43jcb_s909_9lprr0000gn/T/ipykernel_3588/3778700934.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['partner_gender'] = np.where(df['gender'] == 0, 1, 0)


,gender,partner_gender
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1
...,...,...
8373,1,0
8374,1,0
8375,1,0
8376,1,0


### Create a 'Match' column 

This is based off the columns 'dec' and 'dec_0'. 

In [93]:
df['match'] = ((df['dec'] == 1) & 
               (df['dec_o'] == 1)).astype(int)

Verify the logic. If both people say yes (1) then match = 1

In [94]:
print(df[['dec', 'dec_o', 'match']].head(5))

   dec  dec_o  match
0    1      0      0
1    1      0      0
2    1      1      1
3    1      1      1
4    1      1      1


Creating new df with only the columns that are also in HCMST dataset

In [95]:
columbia_filtered = df[target_columns_dict.keys()].copy()

In [96]:
columbia_filtered

,age,age_o,race,race_o,gender,partner_gender,attr,Interracial_relationship,match,dec,dec_o
0,21.0,27.0,ASIAN,WHITE,0,1,6.0,1,0,1,0
1,21.0,22.0,ASIAN,WHITE,0,1,7.0,1,0,1,0
2,21.0,22.0,ASIAN,ASIAN,0,1,5.0,0,1,1,1
3,21.0,23.0,ASIAN,WHITE,0,1,7.0,1,1,1,1
4,21.0,24.0,ASIAN,HISPANIC,0,1,5.0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...
8373,25.0,26.0,WHITE,HISPANIC,1,0,3.0,1,0,0,1
8374,25.0,24.0,WHITE,OTHER,1,0,4.0,1,0,0,0
8375,25.0,29.0,WHITE,HISPANIC,1,0,4.0,1,0,0,0
8376,25.0,22.0,WHITE,ASIAN,1,0,4.0,1,0,0,1


### Bin the age's of the subject and partner to match the format that is in the Stanford HCMST dataset

also create a column if they are in the same age bracket.

In [97]:
shared_bins = [17, 20, 30, 40, 50, 60, 70, 120]
shared_labels = [11, 21, 31, 41, 51, 61, 71]


columbia_filtered['Subject_age_bin'] = pd.cut(columbia_filtered['age'], 
                                        bins=shared_bins, 
                                        labels=shared_labels)

columbia_filtered['Partner_age_bin'] = pd.cut(columbia_filtered['age_o'], 
                                        bins=shared_bins, 
                                        labels=shared_labels)

columbia_filtered['Same_Age_Bracket'] = (columbia_filtered['Subject_age_bin'] == columbia_filtered['Partner_age_bin']).astype(int)

### Rename Columns to match HCTMS

In [ ]:
target_columns_with_bins = {
    'race': "Subject race",
    'race_o': "Partner race",  
    'gender': "Subject gender",
    'partner_gender': "Partner gender", 
    'attr': "Attraction",
    'Interracial_relationship': 'Interracial relationship', 
    'match' : 'match',
    'dec': "Subject dec", 
    'dec_o' : 'Partner dec',
    'Subject_age_bin' : "Subject age",
    'Partner_age_bin' : "Partner age"   
}


columbia_filtered.rename(columns=target_columns_with_bins, inplace=True)

### Drop NA

In [99]:
columbia_filtered = columbia_filtered.dropna()

### Convert to INT64

In [100]:
columbia_filtered

,age,age_o,Subject race,Partner race,Subject gender,Partner gender,Attraction,Interracial relationship,match,Subject dec,Partner dec,Subject age,Partner age,Same_Age_Bracket
0,21.0,27.0,ASIAN,WHITE,0,1,6.0,1,0,1,0,21,21,1
1,21.0,22.0,ASIAN,WHITE,0,1,7.0,1,0,1,0,21,21,1
2,21.0,22.0,ASIAN,ASIAN,0,1,5.0,0,1,1,1,21,21,1
3,21.0,23.0,ASIAN,WHITE,0,1,7.0,1,1,1,1,21,21,1
4,21.0,24.0,ASIAN,HISPANIC,0,1,5.0,1,1,1,1,21,21,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8373,25.0,26.0,WHITE,HISPANIC,1,0,3.0,1,0,0,1,21,21,1
8374,25.0,24.0,WHITE,OTHER,1,0,4.0,1,0,0,0,21,21,1
8375,25.0,29.0,WHITE,HISPANIC,1,0,4.0,1,0,0,0,21,21,1
8376,25.0,22.0,WHITE,ASIAN,1,0,4.0,1,0,0,1,21,21,1


In [ ]:
cols_to_convert = columbia_filtered.columns.drop(['Subject race', 'Partner race'])
columbia_filtered[cols_to_convert] = columbia_filtered[cols_to_convert].round().astype('Int64')

### Remove rows where attraction is 0

In [ ]:
index_to_drop = columbia_filtered[columbia_filtered['Attraction'] == 0].index
columbia_filtered = columbia_filtered.drop(index_to_drop)

In [104]:
columbia_filtered

,age,age_o,Subject race,Partner race,Subject gender,Partner gender,Attraction,Interracial relationship,match,Subject dec,Partner dec,Subject age,Partner age,Same_Age_Bracket
0,21,27,ASIAN,WHITE,0,1,6,1,0,1,0,21,21,1
1,21,22,ASIAN,WHITE,0,1,7,1,0,1,0,21,21,1
2,21,22,ASIAN,ASIAN,0,1,5,0,1,1,1,21,21,1
3,21,23,ASIAN,WHITE,0,1,7,1,1,1,1,21,21,1
4,21,24,ASIAN,HISPANIC,0,1,5,1,1,1,1,21,21,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8373,25,26,WHITE,HISPANIC,1,0,3,1,0,0,1,21,21,1
8374,25,24,WHITE,OTHER,1,0,4,1,0,0,0,21,21,1
8375,25,29,WHITE,HISPANIC,1,0,4,1,0,0,0,21,21,1
8376,25,22,WHITE,ASIAN,1,0,4,1,0,0,1,21,21,1


### Export 

This file will be used/loaded in combining_datasets notebook

In [ ]:
columbia_filtered.to_csv(output_path, index=False)